In [3]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

In [ ]:
from pathlib import Path

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
RAW = ROOT / "data" / "raw" / "reddit.csv"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

def clean(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)        # URLs
    text = re.sub(r"@\w+|\bu/\w+|\br/\w+", " ", text)         # mentions, u/, r/
    text = re.sub(r"[^a-z\s']", " ", text)                    # caractères spéciaux (garde l'apostrophe : négations)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = pd.read_csv(RAW).dropna(subset=["clean_comment", "category"])
df["text"] = df["clean_comment"].map(clean)
df = df[df["text"].str.len() > 0]
# supprime les textes dupliqués ayant des labels contradictoires, puis les doublons exacts
df = df[df.groupby("text")["category"].transform("nunique") == 1]
df = df.drop_duplicates("text").reset_index(drop=True)
df["n_words"] = df["text"].str.split().str.len()

dist = df["category"].value_counts(normalize=True).sort_index()
print(dist.round(3))
print(df["n_words"].describe().round(1))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df["category"].value_counts().sort_index().plot.bar(ax=ax[0], title="Classes")
df["n_words"].clip(upper=df["n_words"].quantile(0.99)).plot.hist(ax=ax[1], bins=40, title="Longueur (mots)")
plt.tight_layout()
plt.savefig(OUT / "eda.png", dpi=120)
plt.show()

train, test = train_test_split(df[["text", "category"]], test_size=0.2, stratify=df["category"], random_state=SEED)

# rééquilibrage du train uniquement (le test garde la distribution réelle)
if dist.max() / dist.min() > 1.5:
    n = train["category"].value_counts().max()
    train = pd.concat([resample(g, replace=True, n_samples=n, random_state=SEED) for _, g in train.groupby("category")])
    train = train.sample(frac=1, random_state=SEED).reset_index(drop=True)

train.to_csv(OUT / "train.csv", index=False)
test.to_csv(OUT / "test.csv", index=False)
print(train["category"].value_counts().to_dict(), test["category"].value_counts().to_dict())
